In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import pandas as pd

# Definindo parâmetros
prob_spread = 0.53  # Probabilidade de propagação do fogo para uma célula vizinha
TIMESTEPS = 200  # Número de passos de tempo
#show_animation = True
show_animation = False

# Estados da célula: 0 = Vegetação, 1 = Em Chamas, 2 = Queimada, 3 = Barreira
EMPTY = 0
BURNING = 1
BURNED = 2
BARRIER = 3

# Carregar o reticulado inicial a partir de um arquivo CSV
forest = pd.read_csv('initial_forest.csv', sep=';', header=None).values
forest_size = forest.shape[0]  # Assume que o CSV é uma matriz quadrada

# Função para propagar o fogo
def spread_fire(forest, prob_spread):
    new_forest = np.copy(forest)

    for i in range(forest_size):
        for j in range(forest_size):
            if forest[i, j] == BURNING:
                # Após queimar, a célula se torna queimada
                new_forest[i, j] = BURNED

                # Verificar células vizinhas
                for x, y in [(i-1, j), (i+1, j), (i, j-1), (i, j+1)]:
                    if 0 <= x < forest_size and 0 <= y < forest_size:
                        if forest[x, y] == EMPTY and np.random.random() < prob_spread:
                            new_forest[x, y] = BURNING
    return new_forest

# Função para contar e imprimir os tipos de células
def print_final_stats(forest):
    empty_cells = np.sum(forest == EMPTY)
    burning_cells = np.sum(forest == BURNING)
    burned_cells = np.sum(forest == BURNED)
    barrier_cells = np.sum(forest == BARRIER)

    print(f"Total de células após {TIMESTEPS} timesteps:")
    print(f"Vegetação (não queimada): {empty_cells}")
    print(f"Células em chamas: {burning_cells}")
    print(f"Células queimadas: {burned_cells}")
    print(f"Barreiras: {barrier_cells}")

# Função de animação para visualizar a propagação do fogo
def update(frame):
    global forest
    forest = spread_fire(forest, prob_spread)
    mat.set_data(forest)

    # No último frame, contar os tipos de células
    if frame == TIMESTEPS - 1:
        print_final_stats(forest)

    return [mat]

# Caso a animação não deva ser mostrada, apenas simular a propagação
if not show_animation:
    for _ in range(TIMESTEPS):
        forest = spread_fire(forest, prob_spread)
    print_final_stats(forest)
else:
    # Se a animação deve ser mostrada, configurar a visualização
    cmap = mcolors.ListedColormap(['green', 'red', 'black', 'gray'])
    bounds = [EMPTY, BURNING, BURNED, BARRIER, BARRIER + 1]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots()
    mat = ax.matshow(forest, cmap=cmap, norm=norm)
    ani = animation.FuncAnimation(fig, update, frames=TIMESTEPS, interval=100, repeat=False)

    plt.show()

Total de células após 200 timesteps:
Vegetação (não queimada): 1227
Células em chamas: 0
Células queimadas: 1173
Barreiras: 100
